# Setup do classificador

Confere GPU/CUDA, extrai um recorte por cromossomo anotado (reaproveitando os splits já usados pelo detector) e valida visualmente uma amostra por classe antes de treinar em `02_train.ipynb`.

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path("../../..").resolve()
sys.path.insert(0, str(REPO_ROOT / "src" / "classification"))
sys.path.insert(0, str(REPO_ROOT / "src" / "preprocessing"))

import torch

print(f"PyTorch: {torch.__version__}")
print(f"GPU disponível: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")


ModuleNotFoundError: No module named 'torch'

Extrai os recortes (`data/processed/classification/{split}/{classe}/*.png` + `manifest.csv`). O split de cada imagem vem das pastas já geradas pelo pipeline de detecção (`data/processed/24_chromosomes_object/images/{train,val,test}`), então recortes da mesma metáfase nunca ficam espalhados entre splits diferentes.

In [ ]:
from extract_crops import run

counts = run(workers=8)
print(f"Imagens: {counts['images']}  Recortes: {counts['crops']} (descartados: {counts['skipped']})")
for split in ("train", "val", "test"):
    print(f"  {split}: {counts[split]}")


Validação visual: uma amostra de cada classe, já com o letterbox 224x224 aplicado (mesmo pré-processamento usado no treino, sem augmentation).

In [ ]:
import cv2
import matplotlib.pyplot as plt
from crops import letterbox
from voc_to_yolo import DENVER_CLASSES

crops_root = REPO_ROOT / "data" / "processed" / "classification"

fig, axes = plt.subplots(4, 6, figsize=(18, 13))
for ax, class_name in zip(axes.flat, DENVER_CLASSES, strict=True):
    sample = next((crops_root / "train" / class_name).glob("*.png"), None)
    if sample is not None:
        crop = cv2.cvtColor(cv2.imread(str(sample)), cv2.COLOR_BGR2GRAY)
        ax.imshow(letterbox(crop), cmap="gray")
    ax.set_title(class_name, fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()
